# Детекция и подсчёт объектов

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Детекция и подсчёт объектов».

## Научная цель

Детекция объектов не просто говорит, что изображено на снимке, а указывает, где именно находится каждый объект и сколько их. Это автоматизирует рутинный визуальный учёт — подсчёт клеток, колоний, частиц, животных — и делает его воспроизводимым.

Полноценный детектор (YOLO, Faster R-CNN) тяжёл для быстрого демо. Поэтому здесь мы на компактных синтетических снимках собственными средствами реализуем учебный детектор «по скользящему окну»: классификатор оценивает фрагменты изображения, а немаксимальное подавление убирает повторные срабатывания. Это показывает все ключевые понятия — рамки, уверенность, IoU, подсчёт — и исполняется на CPU за секунды.

## Интуиция за методом

Представьте лаборанта, который просматривает снимки под микроскопом и ставит карандашную метку на каждую найденную клетку, попутно считая их. Детектор объектов делает то же самое: он находит объекты, обводит каждый прямоугольной рамкой и сообщает, сколько их найдено — и всегда с одинаковым вниманием, независимо от того, первый это снимок или тысячный.

Как это устроено внутри (на интуитивном уровне):
1. Классификатор обучается отличать «окно с объектом в центре» от «окна с фоном» на маленьких вырезках.
2. Это окно «скользит» по всему снимку — проверяет каждую позицию.
3. Там, где уверенность в наличии объекта высока, возникает «кандидат» на рамку.
4. Несколько перекрывающихся кандидатов на одном объекте схлопываются в один — это немаксимальное подавление.

**Ключевые понятия, которые встретятся в блокноте:**
- **Ограничивающая рамка (bounding box)** — прямоугольник, обводящий объект; задаётся координатами (x, y, ширина, высота).
- **Уверенность (confidence score)** — вероятность от 0 до 1, с которой модель считает, что в данном окне есть объект.
- **IoU (Intersection over Union, пересечение над объединением)** — мера совпадения двух рамок: площадь пересечения / площадь объединения. IoU = 1 — идеальное совпадение, IoU = 0 — рамки не перекрываются.
- **NMS (немаксимальное подавление)** — алгоритм, который из нескольких близких рамок оставляет только одну — с наибольшей уверенностью.
- **Порог уверенности** — минимальная уверенность, при которой рамка считается обнаружением. Ниже порога — игнорируем.
- **mAP (mean Average Precision)** — стандартная метрика качества детекторов; в учебном примере мы используем простую ошибку подсчёта вместо неё.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем синтетические «микрофотографии»: на сером шумном фоне разбросаны светлые круглые «частицы». Для каждого снимка известно истинное число частиц и их положения — это позволяет честно проверить и детекцию, и подсчёт.

**Что делает ячейка ниже:** создаёт 60 снимков, показывает два примера с нанесёнными истинными центрами частиц, чтобы вы видели задачу «глазами алгоритма».

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
IMG = 64          # сторона снимка
R = 5             # радиус частицы


def make_image(n_objects):
    """Создаёт снимок с n_objects светлыми частицами."""
    img = 0.25 + 0.05 * rng.standard_normal((IMG, IMG))
    centers = []
    attempts = 0
    while len(centers) < n_objects and attempts < 200:
        attempts += 1
        cx, cy = rng.integers(R + 1, IMG - R - 1, size=2)
        if any((cx - x) ** 2 + (cy - y) ** 2 < (2.2 * R) ** 2
               for x, y in centers):
            continue
        centers.append((cx, cy))
        yy, xx = np.ogrid[:IMG, :IMG]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= R ** 2
        img[mask] += 0.8
    return np.clip(img, 0, 1).astype('float32'), centers


# Набор снимков с разным числом частиц.
images, all_centers = [], []
for _ in range(60):
    n = int(rng.integers(3, 9))
    im, c = make_image(n)
    images.append(im)
    all_centers.append(c)
print(f'Сгенерировано снимков: {len(images)}')
print(f'Частиц на первом снимке: {len(all_centers[0])}')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Показываем два снимка с истинным положением частиц (кружки).
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.4))
for idx, ax in enumerate(axes):
    ax.imshow(images[idx], cmap="gray", vmin=0, vmax=1)
    for cx, cy in all_centers[idx]:
        ax.add_patch(Circle((cx, cy), R, fill=False,
                             edgecolor=VIZ["series"][1], linewidth=1.8))
    ax.set_title(f"Снимок {idx + 1}: {len(all_centers[idx])} частицы")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("Синтетические снимки с отмеченными истинными частицами",
             fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график.** Серый фон с шумом — «пустое поле зрения». Светлые круги — частицы. Зелёные окружности — истинные границы объектов, нанесённые алгоритмом генерации (в реальном эксперименте их рисует разметчик). Именно такие кружки / прямоугольники нужны для обучения детектора.

### Обучающие фрагменты

**Что делает ячейка ниже:** нарезает маленькие квадратные окна из снимков. Окна двух видов:
- **Положительные** (метка 1) — окно центрировано на известной частице; классификатор должен научиться говорить «да, здесь объект».
- **Отрицательные** (метка 0) — окно попало на фон; классификатор должен говорить «нет, здесь пусто».

Это стандартный способ собрать обучающую выборку для детектора на основе скользящего окна.

In [ ]:
WIN = 2 * R + 3      # сторона окна-фрагмента
HALF = WIN // 2


def crop(img, cx, cy):
    """Вырезает квадратное окно с центром (cx, cy)."""
    return img[cy - HALF:cy + HALF + 1, cx - HALF:cx + HALF + 1]


patches, labels = [], []
for img, centers in zip(images[:40], all_centers[:40]):
    # Положительные примеры: окна с частицей в центре.
    for cx, cy in centers:
        p = crop(img, cx, cy)
        if p.shape == (WIN, WIN):
            patches.append(p); labels.append(1)
    # Отрицательные примеры: случайные окна фона.
    for _ in range(len(centers) + 2):
        cx, cy = rng.integers(HALF, IMG - HALF, size=2)
        if all((cx - x) ** 2 + (cy - y) ** 2 > (1.5 * R) ** 2
               for x, y in centers):
            p = crop(img, cx, cy)
            if p.shape == (WIN, WIN):
                patches.append(p); labels.append(0)

patches = np.stack(patches)[:, None, :, :]
labels = np.array(labels, dtype='int64')
print(f'Обучающих фрагментов: {len(patches)} (частица: {labels.sum()}, фон: {(labels == 0).sum()})')

## 4. Применение метода

### Шаг 1. Обучение классификатора фрагментов

**Что делает ячейка ниже:** обучает небольшую свёрточную сеть отличать фрагмент с частицей от фрагмента фона. Это «мозг» детектора: когда окно будет скользить по снимку, именно эта сеть будет выносить вердикт по каждой позиции.

- **CrossEntropyLoss** — функция потерь для задачи классификации (класс 0 или 1).
- **Точность (accuracy) на фрагментах** — доля правильно размеченных окон обучающей выборки; должна расти к 100 %.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


class PatchNet(nn.Module):
    """Классификатор фрагмента: частица в центре или фон."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 12, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(24, 2),
        )

    def forward(self, x):
        return self.net(x)


model = PatchNet()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.CrossEntropyLoss()
Xp = torch.from_numpy(patches.astype('float32'))
yp = torch.from_numpy(labels)

for epoch in range(1, 81):
    model.train()
    opt.zero_grad()
    loss = crit(model(Xp), yp)
    loss.backward()
    opt.step()
    if epoch % 20 == 0:
        acc = (model(Xp).argmax(1) == yp).float().mean().item()
        print(f'Эпоха {epoch:2d}: потеря {loss.item():.4f}, точность на фрагментах {acc:.3f}')

### Шаг 2. Применение детектора: скользящее окно и NMS

**Что делает ячейка ниже:** применяет обученный классификатор ко всем позициям скользящего окна на тестовых снимках, отбирает срабатывания с уверенностью выше порога `conf_thr`, затем применяет немаксимальное подавление (NMS) — схлопывает близкие рамки в одну. Сравнивает число найденных объектов с истинным числом частиц.

**NMS (немаксимальное подавление):** когда одна частица вызывает 10 срабатываний в соседних позициях, NMS оставляет только то, у которого наибольшая уверенность. Порог подавления `nms_dist` задаёт, насколько близко два центра, чтобы считать их одним объектом.

In [ ]:
def detect(img, stride=2, conf_thr=0.7, nms_dist=2.0 * R):
    """Детектор: скользящее окно + немаксимальное подавление.

    Возвращает список (cx, cy, уверенность).
    """
    model.eval()
    windows, coords = [], []
    for cy in range(HALF, IMG - HALF, stride):
        for cx in range(HALF, IMG - HALF, stride):
            windows.append(crop(img, cx, cy))
            coords.append((cx, cy))
    batch = torch.from_numpy(
        np.stack(windows)[:, None].astype('float32'))
    with torch.no_grad():
        prob = torch.softmax(model(batch), dim=1)[:, 1].numpy()
    # Отбираем окна с высокой уверенностью.
    cand = [(cx, cy, p) for (cx, cy), p in zip(coords, prob)
            if p >= conf_thr]
    cand.sort(key=lambda t: -t[2])
    # Немаксимальное подавление: гасим близкие срабатывания.
    kept = []
    for cx, cy, p in cand:
        if all((cx - kx) ** 2 + (cy - ky) ** 2 > nms_dist ** 2
               for kx, ky, _ in kept):
            kept.append((cx, cy, p))
    return kept


# Проверка подсчёта на отложенных снимках (не участвовали в обучении).
test_imgs = images[40:]
test_centers = all_centers[40:]
true_counts, pred_counts = [], []
for img, centers in zip(test_imgs, test_centers):
    det = detect(img)
    true_counts.append(len(centers))
    pred_counts.append(len(det))
true_counts = np.array(true_counts)
pred_counts = np.array(pred_counts)
mae = np.abs(true_counts - pred_counts).mean()
print(f'Средняя ошибка подсчёта на {len(test_imgs)} снимках: {mae:.2f} частицы')

### Визуализация: «до детекции — после детекции»

Три панели рядом — ключевой наглядный эксперимент:
1. Исходный снимок (без разметки) — то, что поступает на вход детектора.
2. Снимок с истинными рамками (что должно быть найдено) — эталон.
3. Снимок с предсказанными рамками (что нашёл детектор) — результат.

Правый график — сравнение истинного и предсказанного числа объектов по всем тестовым снимкам.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

demo_img = test_imgs[0]
demo_true = test_centers[0]
det = detect(demo_img)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6),
                         gridspec_kw={"wspace": 0.08})

# Панель 1: исходный снимок.
axes[0].imshow(demo_img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Снимок (вход)", fontsize=12)
axes[0].set_xticks([]); axes[0].set_yticks([]); axes[0].grid(False)

# Панель 2: истинные положения частиц (зелёные рамки — эталон).
axes[1].imshow(demo_img, cmap="gray", vmin=0, vmax=1)
for cx, cy in demo_true:
    axes[1].add_patch(Rectangle((cx - R, cy - R), 2 * R, 2 * R,
                      fill=False, edgecolor=VIZ["series"][1], linewidth=2))
axes[1].set_title(f"Эталон: {len(demo_true)} частицы (зелёные рамки)",
                  fontsize=12)
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].grid(False)

# Панель 3: результат детектора (оранжевые рамки — предсказание).
axes[2].imshow(demo_img, cmap="gray", vmin=0, vmax=1)
for cx, cy, p in det:
    axes[2].add_patch(Rectangle((cx - R, cy - R), 2 * R, 2 * R,
                      fill=False, edgecolor=VIZ["series"][2], linewidth=2))
    axes[2].text(cx - R, cy - R - 2, f"{p:.2f}", fontsize=7,
                 color=VIZ["series"][2])
axes[2].set_title(f"Детектор: найдено {len(det)} (оранжевые рамки + уверенность)",
                  fontsize=12)
axes[2].set_xticks([]); axes[2].set_yticks([]); axes[2].grid(False)

fig.suptitle("До и после детекции на одном снимке", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# График точности подсчёта по всем тестовым снимкам.
fig2, ax2 = plt.subplots(figsize=(7.0, 5.4))
ax2.scatter(true_counts, pred_counts, color=VIZ["series"][0],
            alpha=0.75, edgecolor="white", linewidth=0.5, s=60)
lim = [true_counts.min() - 1, true_counts.max() + 1]
ax2.plot(lim, lim, color=VIZ["series"][3], linestyle="--",
         label="Идеальный счёт (предсказано = истинно)")
ax2.set_title("Точность подсчёта: предсказано vs. истинно")
ax2.set_xlabel("Истинное число частиц")
ax2.set_ylabel("Найдено детектором")
ax2.legend(loc="upper left")
fig2.tight_layout()
plt.show()

print(f"Средняя абсолютная ошибка подсчёта: {mae:.2f} частицы")

**Как читать эти графики.**

- **Три панели «до и после»:** сравните среднюю и правую панели — насколько оранжевые рамки детектора совпадают с зелёными эталонными рамками. Цифры уверенности (0.xx) над рамками показывают, насколько модель «уверена» в каждой находке. Пропущенная частица — ложноотрицательный результат; лишняя рамка на фоне — ложноположительный.

- **График «предсказано vs. истинно»:** каждая точка — один тестовый снимок. Если точка лежит на пунктирной диагонали — детектор ошибся ровно на 0 частиц. Точки выше диагонали — детектор нашёл лишние объекты (ложные срабатывания). Точки ниже — пропустил реальные. Хороший детектор: облако точек плотно вдоль диагонали.

## 5. Интерпретация результата

- **Рамки на снимке** показывают, что именно детектор счёл объектом. Визуальная проверка обязательна: она выявляет ложные срабатывания и пропуски.
- **График точности подсчёта**: точки вдоль диагонали означают корректный учёт. Систематическое занижение типично для плотно расположенных, сливающихся объектов — там нужны модели плотности.
- **Порог уверенности** (`conf_thr`) и **порог подавления** (`nms_dist`) напрямую влияют на счёт: их калибруют на отдельной валидации, а не подбирают по тесту.
- Реальные детекторы оценивают метрикой mAP при разных порогах перекрытия рамок (IoU); здесь для наглядности взята ошибка подсчёта.

Для научных задач берут готовые детекторы (YOLO, Faster R-CNN) и переносят обучение с предобученных весов — это снижает требования к разметке.

## 6. Попробуйте на своих данных

Для собственного учёта объектов:

1. Подготовьте снимки и разметку — координаты центров либо рамки объектов.
2. Для простых однотипных объектов подход «окно + классификатор» из этого блокнота переносится напрямую: меняются `IMG`, `R`, `WIN`.
3. Для сложных сцен используйте готовый детектор, например `torchvision.models.detection.fasterrcnn_resnet50_fpn` с предобученными весами, дообучая его на своих рамках.
4. Порог уверенности и порог подавления подбирайте на валидации.

## 5а. Поэкспериментируйте сами

Измените один параметр, перезапустите ячейки обнаружения (раздел 4) и посмотрите, как изменится качество:

| Что изменить | Где | Ожидаемый эффект |
|---|---|---|
| `conf_thr=0.9` вместо 0.7 | функция `detect` | Меньше ложных срабатываний, но больше пропусков |
| `conf_thr=0.5` | функция `detect` | Больше рамок, в том числе ложных |
| `nms_dist=R` вместо `2.0 * R` | функция `detect` | Рядом стоящие частицы могут задвоиться |
| `stride=1` вместо 2 | функция `detect` | Точнее, но медленнее (проверяет больше позиций) |

In [ ]:
# --- Шаблон применения готового детектора ---
# import torch
# from torchvision.models.detection import fasterrcnn_resnet50_fpn
#
# net = fasterrcnn_resnet50_fpn(weights='DEFAULT').eval()
# # image_tensor: float-тензор (3, H, W) со значениями в [0, 1]
# with torch.no_grad():
#     out = net([image_tensor])[0]
# keep = out['scores'] > 0.7        # порог уверенности
# print('Найдено объектов:', int(keep.sum()))


## 7. Краткие выводы

- Детекция объектов = нахождение + локализация + подсчёт. Она отвечает не просто «что изображено», а «где именно и сколько».
- Принцип скользящего окна — основа большинства детекторов: классификатор оценивает каждый фрагмент изображения.
- NMS (немаксимальное подавление) — обязательный шаг постобработки: убирает дублирующие рамки на одном объекте.
- Порог уверенности и порог NMS напрямую влияют на баланс ложных срабатываний и пропусков — их нужно подбирать на валидационной выборке, а не по тестовой.
- Для реальных задач используйте готовые детекторы (YOLO, Faster R-CNN) с переносом обучения — они требуют значительно меньше своей разметки.
- Для плотно слипшихся объектов обычная детекция занижает счёт; в таких случаях применяют модели оценки плотности.